## Goal

In this sprint, we move from exploratory analysis toward an actual recommendation pipeline. We generate candidate movies, build content-based genre representations, create user preference profiles, and develop a baseline scoring method for ranking and recommending movies to individual users. We also evaluate the limitations of this baseline and identify which additional signals may be needed to improve recommendation quality.

In [1]:
# Imports
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [2]:
# Load / Prepare Data
users_df = pd.read_csv('../data/raw/users.dat', sep='::', engine='python', header=None, names=['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code'])
ratings_df = pd.read_csv('../data/raw/ratings.dat', sep='::', engine='python', header=None, names=['UserID', 'MovieID', 'Rating', 'Timestamp'])
movies_df = pd.read_csv('../data/raw/movies.dat', sep='::', engine='python', header=None, names=['MovieID', 'Title', 'Genres'], encoding='latin-1')

In [3]:
ratings_df['TimestampDate'] = pd.to_datetime(ratings_df['Timestamp'], unit='s')

In [4]:
ratings_df['YearMonth'] = ratings_df['TimestampDate'].dt.to_period('M')

In [5]:
# Baseline Recommendation Score
movie_rating_count = ratings_df.groupby('MovieID')['Rating'].size()
movie_average_rating = ratings_df.groupby('MovieID')['Rating'].mean()
m = movie_rating_count.quantile(0.75)
m

np.float64(350.0)

- Movie with v << 350 → has little evidence → its score is more likely to be pulled towards the Global Average.
- Movie with v ≈ 350 → falls between the two.
- Movie with v >> 350 → has a lot of evidence → its actual score gets more weight.

In [6]:
C = ratings_df['Rating'].mean()
C

np.float64(3.581564453029317)

In [7]:
v = movie_rating_count
R = movie_average_rating

In [8]:
weighted_rating = (v / (v+m)) * R + (m / (v+m)) * C
weighted_rating

MovieID
1       4.065327
2       3.327828
3       3.255492
4       3.302976
5       3.318185
          ...   
3948    3.620089
3949    3.829583
3950    3.592940
3951    3.614225
3952    3.686379
Name: Rating, Length: 3706, dtype: float64

In [9]:
# Top 10 popular movies
weighted_rating.sort_values(ascending=False).head(10)

MovieID
318     4.422409
858     4.396637
527     4.387923
1198    4.368208
50      4.363595
260     4.362331
2762    4.303506
750     4.272887
912     4.268721
593     4.259750
Name: Rating, dtype: float64

In [10]:
movie_ranking_df = pd.DataFrame({
    'Average Rating': movie_average_rating,
    'Rating Count': movie_rating_count,
    'Weighted Rating': weighted_rating
})
movie_ranking_df

,Average Rating,Rating Count,Weighted Rating
MovieID,,,
1,4.146846,2077,4.065327
2,3.201141,701,3.327828
3,3.016736,478,3.255492
4,2.729412,170,3.302976
5,3.006757,296,3.318185
...,...,...,...
3948,3.635731,862,3.620089
3949,4.115132,304,3.829583
3950,3.666667,54,3.592940


In [11]:
movie_ranking_with_metadata_df = movie_ranking_df.merge(
    movies_df,
    on='MovieID',
    how='left'
)
movie_ranking_with_metadata_df.sort_values(by='Weighted Rating', ascending=False).head(10)

,MovieID,Average Rating,Rating Count,Weighted Rating,Title,Genres
309,318,4.554558,2227,4.422409,"Shawshank Redemption, The (1994)",Drama
802,858,4.524966,2223,4.396637,"Godfather, The (1972)",Action|Crime|Drama
513,527,4.510417,2304,4.387923,Schindler's List (1993),Drama|War
1108,1198,4.477725,2514,4.368208,Raiders of the Lost Ark (1981),Action|Adventure
49,50,4.517106,1783,4.363595,"Usual Suspects, The (1995)",Crime|Thriller
253,260,4.453694,2991,4.362331,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Fantasy|Sci-Fi
2557,2762,4.406263,2459,4.303506,"Sixth Sense, The (1999)",Thriller
713,750,4.449890,1367,4.272887,Dr. Strangelove or: How I Learned to Stop Worr...,Sci-Fi|War
851,912,4.412822,1669,4.268721,Casablanca (1942),Drama|Romance|War
579,593,4.351823,2578,4.259750,"Silence of the Lambs, The (1991)",Drama|Thriller


In [12]:
movie_ranking_df.sort_values(
    by='Average Rating',
    ascending=False
).head(10)

,Average Rating,Rating Count,Weighted Rating
MovieID,,,
3280,5.0,1,3.585606
3382,5.0,1,3.585606
3172,5.0,1,3.585606
3233,5.0,2,3.589624
787,5.0,3,3.593619
1830,5.0,1,3.585606
989,5.0,1,3.585606
3656,5.0,1,3.585606
3607,5.0,1,3.585606


In [13]:
# Recommendation Candidate Generation
candidate_threshold = movie_rating_count.median().round()
candidate_threshold

np.float64(124.0)

Use the median movie rating count as an initial candidate-generation threshold because the movie interaction distribution is strongly right-skewed.

In [14]:
candidate_movies_df = movie_ranking_with_metadata_df.query(f'`Rating Count` >= {candidate_threshold}')
candidate_movies_df

,MovieID,Average Rating,Rating Count,Weighted Rating,Title,Genres
0,1,4.146846,2077,4.065327,Toy Story (1995),Animation|Children's|Comedy
1,2,3.201141,701,3.327828,Jumanji (1995),Adventure|Children's|Fantasy
2,3,3.016736,478,3.255492,Grumpier Old Men (1995),Comedy|Romance
3,4,2.729412,170,3.302976,Waiting to Exhale (1995),Comedy|Drama
4,5,3.006757,296,3.318185,Father of the Bride Part II (1995),Comedy
...,...,...,...,...,...,...
3685,3932,3.750000,232,3.648707,"Invisible Man, The (1933)",Horror|Sci-Fi
3690,3937,2.940741,135,3.403191,Runaway (1984),Sci-Fi|Thriller
3701,3948,3.635731,862,3.620089,Meet the Parents (2000),Comedy
3702,3949,4.115132,304,3.829583,Requiem for a Dream (2000),Drama


Using the median movie interaction count as the initial candidate threshold reduced the movie pool from 3,706 to 1,853 candidates, retaining approximately 50% of the available movies. This provides a balanced initial filtering strategy that removes low-interaction items without aggressively excluding potentially relevant movies.

In [15]:
user_id = 1
user_rated_movies = ratings_df.loc[
    ratings_df['UserID'] == user_id,
    'MovieID'
]

In [16]:
user_candidates_df = candidate_movies_df[
    ~candidate_movies_df['MovieID'].isin(user_rated_movies)
]

In [17]:
user_candidates_df

,MovieID,Average Rating,Rating Count,Weighted Rating,Title,Genres
1,2,3.201141,701,3.327828,Jumanji (1995),Adventure|Children's|Fantasy
2,3,3.016736,478,3.255492,Grumpier Old Men (1995),Comedy|Romance
3,4,2.729412,170,3.302976,Waiting to Exhale (1995),Comedy|Drama
4,5,3.006757,296,3.318185,Father of the Bride Part II (1995),Comedy
5,6,3.878723,940,3.798099,Heat (1995),Action|Crime|Thriller
...,...,...,...,...,...,...
3685,3932,3.750000,232,3.648707,"Invisible Man, The (1933)",Horror|Sci-Fi
3690,3937,2.940741,135,3.403191,Runaway (1984),Sci-Fi|Thriller
3701,3948,3.635731,862,3.620089,Meet the Parents (2000),Comedy
3702,3949,4.115132,304,3.829583,Requiem for a Dream (2000),Drama


User-specific filtering should be applied after global candidate filtering, because not every previously rated movie belongs to the global candidate pool.

In [18]:
genre_matrix = movies_df['Genres'].str.get_dummies('|')

In [19]:
genre_matrix.shape

(3883, 18)

In [20]:
genre_matrix.head()

,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [21]:
# Content-Based Signal
genre_similarity = cosine_similarity(genre_matrix)
genre_similarity.shape

(3883, 3883)

In [22]:
movie_index = movies_df.index[movies_df['MovieID'] == 1][0]
similarities = genre_similarity[movie_index]

In [23]:
similar_movies_indices = similarities.argsort()[::-1]

In [24]:
top_similar_indices = similar_movies_indices[:11]
top_similar_movies = movies_df.iloc[top_similar_indices][
    ['MovieID', 'Title', 'Genres']
].copy()
top_similar_movies = top_similar_movies[
    top_similar_movies['MovieID'] != 1
]
top_similar_movies['Similarity'] = similarities[top_similar_movies.index]
top_similar_movies

,MovieID,Title,Genres,Similarity
2072,2141,"American Tail, An (1986)",Animation|Children's|Comedy,1.000000
3682,3751,Chicken Run (2000),Animation|Children's|Comedy,1.000000
3685,3754,"Adventures of Rocky and Bullwinkle, The (2000)",Animation|Children's|Comedy,1.000000
1050,1064,Aladdin and the King of Thieves (1996),Animation|Children's|Comedy,1.000000
3542,3611,Saludos Amigos (1943),Animation|Children's|Comedy,1.000000
2285,2354,"Rugrats Movie, The (1998)",Animation|Children's|Comedy,1.000000
2073,2142,"American Tail: Fievel Goes West, An (1991)",Animation|Children's|Comedy,1.000000
3045,3114,Toy Story 2 (1999),Animation|Children's|Comedy,1.000000
2286,2355,"Bug's Life, A (1998)",Animation|Children's|Comedy,1.000000
2009,2078,"Jungle Book, The (1967)",Animation|Children's|Comedy|Musical,0.866025


Genre-based cosine similarity successfully identifies movies with similar genre profiles. Movies sharing identical genre combinations receive a similarity score of 1.0, while partial genre overlap produces lower similarity scores.

Genre similarity alone cannot capture differences between movies that share the same genres.

In [25]:
# User Preference Profile
user_movie_df = ratings_df.merge(
    movies_df[['MovieID', 'Title', 'Genres']],
    on='MovieID',
    how='left'
)

In [26]:
user_movie_df['Genres'] = (
    user_movie_df['Genres']
    .str.split('|')
)
user_movie_df = user_movie_df.explode('Genres')

In [27]:
user_movie_df.head()

,UserID,MovieID,Rating,Timestamp,TimestampDate,YearMonth,Title,Genres
0,1,1193,5,978300760,2000-12-31 22:12:40,2000-12,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,978302109,2000-12-31 22:35:09,2000-12,James and the Giant Peach (1996),Animation
1,1,661,3,978302109,2000-12-31 22:35:09,2000-12,James and the Giant Peach (1996),Children's
1,1,661,3,978302109,2000-12-31 22:35:09,2000-12,James and the Giant Peach (1996),Musical
2,1,914,3,978301968,2000-12-31 22:32:48,2000-12,My Fair Lady (1964),Musical


In [28]:
user_genre_profile_df = (
    user_movie_df
    .groupby(['UserID', 'Genres'])['Rating']
    .agg(['mean', 'count'])
    .reset_index()
)

In [29]:
user_genre_profile_df = user_genre_profile_df.rename(
    columns={
        'mean': 'Average Rating',
        'count': 'Rating Count'
    }
)

In [30]:
R = user_genre_profile_df['Average Rating']
v = user_genre_profile_df['Rating Count']
C = ratings_df['Rating'].mean()
m = np.median(v)
preference_score = (v / (v+m)) * R + (m / (v+m)) * C

In [31]:
user_genre_profile_df['Preference Score'] = preference_score

In [32]:
user_genre_profile_df

,UserID,Genres,Average Rating,Rating Count,Preference Score
0,1,Action,4.200000,5,3.819424
1,1,Adventure,4.000000,5,3.742501
2,1,Animation,4.111111,18,3.948174
3,1,Children's,4.250000,20,4.059018
4,1,Comedy,4.142857,14,3.938751
...,...,...,...,...,...
93882,6040,Romance,3.488889,45,3.502878
93883,6040,Sci-Fi,3.473684,38,3.492446
93884,6040,Thriller,3.926829,41,3.870460
93885,6040,War,3.695652,23,3.666210


In [33]:
user_1_preference = user_genre_profile_df[user_genre_profile_df['UserID'] == 1].sort_values(by='Preference Score', ascending=False)
user_1_preference

,UserID,Genres,Average Rating,Rating Count,Preference Score
6,1,Drama,4.428571,21,4.194914
3,1,Children's,4.250000,20,4.059018
8,1,Musical,4.285714,14,4.029660
2,1,Animation,4.111111,18,3.948174
4,1,Comedy,4.142857,14,3.938751
12,1,War,5.000000,2,3.865252
0,1,Action,4.200000,5,3.819424
10,1,Sci-Fi,4.333333,3,3.786592
1,1,Adventure,4.000000,5,3.742501
7,1,Fantasy,4.000000,3,3.695683


We use user-level Min-Max normalization to transform genre preference scores into relative preference weights before calculating weighted movie scores.

In [34]:
# Top-N Recommendation
scaler = MinMaxScaler()

user_1_preference['Preference Weight'] = scaler.fit_transform(
    user_1_preference[['Preference Score']]
).ravel()
user_1_preference

,UserID,Genres,Average Rating,Rating Count,Preference Score,Preference Weight
6,1,Drama,4.428571,21,4.194914,1.000000
3,1,Children's,4.250000,20,4.059018,0.769723
8,1,Musical,4.285714,14,4.029660,0.719974
2,1,Animation,4.111111,18,3.948174,0.581895
4,1,Comedy,4.142857,14,3.938751,0.565928
12,1,War,5.000000,2,3.865252,0.441382
0,1,Action,4.200000,5,3.819424,0.363727
10,1,Sci-Fi,4.333333,3,3.786592,0.308093
1,1,Adventure,4.000000,5,3.742501,0.233380
7,1,Fantasy,4.000000,3,3.695683,0.154047


In [35]:
user_1_candidates_genre_df = user_candidates_df.copy()

user_1_candidates_genre_df['Genres'] = (
    user_1_candidates_genre_df['Genres']
    .str.split('|')
)

user_1_candidates_genre_df = user_1_candidates_genre_df.explode('Genres')

In [36]:
user_1_movie_preference_df = user_1_candidates_genre_df.merge(
    user_1_preference[['Genres', 'Preference Score', 'Preference Weight']],
    on='Genres',
    how='left'
)

In [37]:
user_1_movie_preference_df.head()

,MovieID,Average Rating,Rating Count,Weighted Rating,Title,Genres,Preference Score,Preference Weight
0,2,3.201141,701,3.327828,Jumanji (1995),Adventure,3.742501,0.233380
1,2,3.201141,701,3.327828,Jumanji (1995),Children's,4.059018,0.769723
2,2,3.201141,701,3.327828,Jumanji (1995),Fantasy,3.695683,0.154047
3,3,3.016736,478,3.255492,Grumpier Old Men (1995),Comedy,3.938751,0.565928
4,3,3.016736,478,3.255492,Grumpier Old Men (1995),Romance,3.618037,0.022474


In [38]:
simple_movie_score = (
    user_1_movie_preference_df
    .groupby('MovieID')['Preference Score']
    .mean()
)

In [39]:
user_1_movie_preference_df['Weighted Preference'] = (
    user_1_movie_preference_df['Preference Score']
    * user_1_movie_preference_df['Preference Weight']
)

In [40]:
weighted_numerator = (
    user_1_movie_preference_df
    .groupby('MovieID')['Weighted Preference']
    .sum()
)
weighted_denominator = (
    user_1_movie_preference_df
    .groupby('MovieID')['Preference Weight']
    .sum()
)

In [41]:
weighted_movie_score = weighted_numerator / weighted_denominator
weighted_movie_score

MovieID
2       3.946812
3       3.926501
4       4.102337
5       3.938751
6       3.785535
          ...   
3932    3.786592
3937    3.786592
3948    3.938751
3949    4.194914
3952    4.194914
Length: 1801, dtype: float64

In [42]:
user_1_movie_scores_df = pd.DataFrame({
    'Simple Score': simple_movie_score,
    'Weighted Score': weighted_movie_score
})
user_1_movie_scores_df = (
    user_1_movie_scores_df
    .reset_index()
    .merge(
        user_candidates_df[['MovieID', 'Title', 'Genres']],
        on='MovieID',
        how='left'
    )
)

In [43]:
top_n_recommendations = user_1_movie_scores_df.sort_values(by='Weighted Score', ascending=False).head(10)
top_n_simple = user_1_movie_scores_df.sort_values(by='Simple Score', ascending=False).head(10)

In [44]:
top_n_recommendations

,MovieID,Simple Score,Weighted Score,Title,Genres
9,14,4.194914,4.194914,Nixon (1995),Drama
1800,3952,3.899844,4.194914,"Contender, The (2000)",Drama|Thriller
1775,3871,4.194914,4.194914,Shane (1953),Drama|Western
808,1840,4.194914,4.194914,He Got Game (1998),Drama
815,1873,4.194914,4.194914,"Misérables, Les (1998)",Drama
830,1913,4.194914,4.194914,Picnic at Hanging Rock (1975),Drama|Mystery
763,1694,4.194914,4.194914,"Apostle, The (1997)",Drama
764,1699,4.194914,4.194914,"Butcher Boy, The (1998)",Drama
1727,3747,4.194914,4.194914,Jesus' Son (1999),Drama
1745,3788,4.194914,4.194914,Blowup (1966),Drama|Mystery


The baseline recommendation score is primarily driven by genre preference and cannot distinguish between movies sharing the same genre composition.

In [45]:
top_n_simple

,MovieID,Simple Score,Weighted Score,Title,Genres
1775,3871,4.194914,4.194914,Shane (1953),Drama|Western
9,14,4.194914,4.194914,Nixon (1995),Drama
1799,3949,4.194914,4.194914,Requiem for a Dream (2000),Drama
1766,3844,4.194914,4.194914,Steel Magnolias (1989),Drama
502,1177,4.194914,4.194914,Enchanted April (1991),Drama
501,1176,4.194914,4.194914,"Double Life of Veronique, The (La Double Vie d...",Drama
499,1173,4.194914,4.194914,"Cook the Thief His Wife & Her Lover, The (1989)",Drama
496,1150,4.194914,4.194914,"Return of Martin Guerre, The (Retour de Martin...",Drama
491,1132,4.194914,4.194914,Manon of the Spring (Manon des sources) (1986),Drama
490,1131,4.194914,4.194914,Jean de Florette (1986),Drama


The current baseline should not assign the same score to two movies simply because they share the same genre. Movie-level signals such as the overall rating and rating count should also be incorporated into the recommendation score.

A movie with a high average rating based on a large number of ratings provides stronger evidence of quality than a movie with the same average rating based on only a few ratings. Therefore, both the movie's average rating and its rating count should be considered.

The recommendation score can be viewed as a combination of two perspectives: the user's preference toward a genre and the movie's overall performance across the entire user population.

In [46]:
# Evaluate the Baseline

For evaluation, we define a relevant interaction as a rating of 4 or higher. Ratings below 4 are considered non-relevant. This threshold allows us to evaluate whether the recommender can retrieve movies that the user actually rated positively.

In [47]:
ratings_df.duplicated(['UserID', 'MovieID']).sum()

np.int64(0)

In [48]:
relevance_threshold = 4
ratings_sorted = ratings_df.sort_values(['UserID', 'Timestamp']).copy()

In [49]:
user_counts = ratings_sorted.groupby('UserID')['MovieID'].transform('size')

In [50]:
user_positions = ratings_sorted.groupby('UserID').cumcount()

In [51]:
test_mask = user_positions >= (user_counts * 0.8)

test_df = ratings_sorted[test_mask].copy()

train_df = ratings_sorted[~test_mask].copy()

We use a temporal 80/20 train-test split for each user. Since the dataset guarantees at least 20 ratings per user, each user contributes at least four interactions to the test set while retaining sufficient historical interactions for building the user profile.

In [52]:
test_relevant_df = test_df[
    test_df['Rating'] >= relevance_threshold
].copy()

In [53]:
relevant_count_per_user = (
    test_df
    .assign(
        Relevant=test_df['Rating'] >= relevance_threshold
    )
    .groupby('UserID')['Relevant']
    .sum()
)

In [54]:
relevant_count_per_user.describe()

count    6040.000000
mean       17.078146
std        19.441963
min         0.000000
25%         5.000000
50%        10.000000
75%        22.000000
max       229.000000
Name: Relevant, dtype: float64

In [55]:
(relevant_count_per_user == 0).sum()

np.int64(84)

In [56]:
evaluation_users = relevant_count_per_user[
    relevant_count_per_user > 0
].index

In [57]:
evaluation_users.nunique()

5956

84 users had no relevant interactions in the test set and were excluded from ranking-metric evaluation because Recall is undefined when the number of relevant items is zero.

In [58]:
train_user_genre_profile_df = (
    train_df
    .merge(
        movies_df[['MovieID', 'Genres']],
        on='MovieID',
        how='left'
    )
)

In [59]:
train_user_genre_profile_df['Genres'] = (
    train_user_genre_profile_df['Genres']
    .str.split('|')
)

train_user_genre_profile_df = (
    train_user_genre_profile_df
    .explode('Genres')
)

In [60]:
train_user_genre_profile_df = (
    train_user_genre_profile_df
    .groupby(['UserID', 'Genres'])['Rating']
    .agg(['mean', 'count'])
    .reset_index()
)

In [61]:
C = train_df['Rating'].mean()
m = np.median(train_user_genre_profile_df['count'])

R = train_user_genre_profile_df['mean']
v = train_user_genre_profile_df['count']

train_user_genre_profile_df['Preference Score'] = (
    (v / (v + m)) * R +
    (m / (v + m)) * C
)

In [62]:
train_user_genre_profile_df.head()

,UserID,Genres,mean,count,Preference Score
0,1,Action,4.200000,5,3.860053
1,1,Adventure,4.000000,4,3.756422
2,1,Animation,4.125000,8,3.888043
3,1,Children's,4.272727,11,4.017813
4,1,Comedy,4.111111,9,3.895040


In [63]:
train_user_genre_profile_df[
    train_user_genre_profile_df['UserID'] == 1
].sort_values(
    'Preference Score',
    ascending=False
)

,UserID,Genres,mean,count,Preference Score
6,1,Drama,4.428571,21,4.225737
8,1,Musical,4.300000,10,4.018861
3,1,Children's,4.272727,11,4.017813
12,1,War,5.000000,2,3.924515
4,1,Comedy,4.111111,9,3.895040
2,1,Animation,4.125000,8,3.888043
0,1,Action,4.200000,5,3.860053
10,1,Sci-Fi,4.333333,3,3.832064
1,1,Adventure,4.000000,4,3.756422
7,1,Fantasy,4.000000,3,3.732064


In [64]:
train_user_genre_profile_df['Preference Score Scaled'] = (
    scaler.fit_transform(
        train_user_genre_profile_df[['Preference Score']]
    )
)

In [65]:
train_movie_stats_df = (
    train_df
    .groupby('MovieID')['Rating']
    .agg(
        Average_Rating='mean',
        Rating_Count='count'
    )
    .reset_index()
)

C_movie = train_df['Rating'].mean()
m_movie = train_movie_stats_df['Rating_Count'].median()

train_movie_stats_df['Weighted_Rating'] = (
    (
        train_movie_stats_df['Rating_Count']
        /
        (
            train_movie_stats_df['Rating_Count']
            + m_movie
        )
    )
    * train_movie_stats_df['Average_Rating']
    +
    (
        m_movie
        /
        (
            train_movie_stats_df['Rating_Count']
            + m_movie
        )
    )
    * C_movie
)

In [66]:
movie_quality_scaler = MinMaxScaler()

train_movie_stats_df['Weighted Rating Scaled'] = (
    movie_quality_scaler.fit_transform(
        train_movie_stats_df[['Weighted_Rating']]
    ).ravel()
)

In [67]:
candidate_movies_df = (
    movies_df
    .merge(
        train_movie_stats_df,
        on='MovieID',
        how='inner'
    )
)

candidate_movies_df = candidate_movies_df[
    candidate_movies_df['Rating_Count'] >= candidate_threshold
].copy()

In [68]:
user_id = 1
user_preference = (
    train_user_genre_profile_df[
        train_user_genre_profile_df['UserID'] == user_id
    ]
    .sort_values(
        by='Preference Score Scaled',
        ascending=False
    )
)

In [69]:
user_train_rated_movies = train_df.loc[
    train_df['UserID'] == user_id,
    'MovieID'
].unique()

In [70]:
user_candidates_df = candidate_movies_df[
    ~candidate_movies_df['MovieID'].isin(user_train_rated_movies)
].copy()

In [71]:
user_movie_preference_df = (
    user_candidates_df
    .assign(Genres=user_candidates_df['Genres'].str.split('|'))
    .explode('Genres')
)

In [72]:
candidate_movies_df = (
    movies_df
    .merge(
        train_movie_stats_df,
        on='MovieID',
        how='inner'
    )
)

In [73]:
user_movie_preference_df = user_movie_preference_df.merge(
    user_preference[
        ['Genres', 'Preference Score', 'Preference Score Scaled']
    ],
    on='Genres',
    how='left'
)
user_movie_preference_df

,MovieID,Title,Genres,Average_Rating,Rating_Count,Weighted_Rating,Weighted Rating Scaled,Preference Score,Preference Score Scaled
0,1,Toy Story (1995),Animation,4.139295,1845,4.114499,0.827074,3.888043,0.737370
1,1,Toy Story (1995),Children's,4.139295,1845,4.114499,0.827074,4.017813,0.772218
2,1,Toy Story (1995),Comedy,4.139295,1845,4.114499,0.827074,3.895040,0.739249
3,2,Jumanji (1995),Adventure,3.200351,569,3.258374,0.464979,3.756422,0.702026
4,2,Jumanji (1995),Children's,3.200351,569,3.258374,0.464979,4.017813,0.772218
...,...,...,...,...,...,...,...,...,...
3089,3932,"Invisible Man, The (1933)",Sci-Fi,3.731579,190,3.694275,0.649342,3.832064,0.722338
3090,3948,Meet the Parents (2000),Comedy,3.671848,579,3.664360,0.636689,3.895040,0.739249
3091,3949,Requiem for a Dream (2000),Drama,4.178571,140,3.955972,0.760025,4.225737,0.828052
3092,3952,"Contender, The (2000)",Drama,3.803150,254,3.753715,0.674482,4.225737,0.828052


In [74]:
user_movie_preference_df['Final Score'] = (
    2
    * user_movie_preference_df['Preference Score Scaled']
    * user_movie_preference_df['Weighted Rating Scaled']
    /
    (
        user_movie_preference_df['Preference Score Scaled']
        + user_movie_preference_df['Weighted Rating Scaled']
    )
)

In [86]:
movie_scores_df = (
    user_movie_preference_df
    .groupby('MovieID')
    .agg(
        Simple_Score=('Preference Score Scaled', 'mean'),
        Final_Score=('Final Score', 'mean')
    )
    .reset_index()
)

In [87]:
movie_scores_df = (
    movie_scores_df
    .merge(
        user_candidates_df[
            [
                'MovieID',
                'Title',
                'Genres',
                'Average_Rating',
                'Rating_Count',
                'Weighted_Rating'
            ]
        ],
        on='MovieID',
        how='left'
    )
)

In [75]:
simple_movie_score = (
    user_movie_preference_df
    .groupby('MovieID')['Preference Score Scaled']
    .mean()
)

In [76]:
user_movie_preference_df['Weighted Preference'] = (
    user_movie_preference_df['Preference Score']
    * user_movie_preference_df['Preference Score Scaled']
)

In [77]:
weighted_numerator = (
    user_movie_preference_df
    .groupby('MovieID')['Weighted Preference']
    .sum()
)

In [78]:
weighted_denominator = (
    user_movie_preference_df
    .groupby('MovieID')['Preference Score Scaled']
    .sum()
)

In [79]:
weighted_movie_score = (
    weighted_numerator
    / weighted_denominator
)

In [80]:
user_movie_scores_df = pd.DataFrame({
    'Simple Score': simple_movie_score,
    'Weighted Score': weighted_movie_score
})

In [81]:
user_movie_scores_df = (
    user_movie_scores_df
    .reset_index()
    .merge(
        user_candidates_df[['MovieID', 'Title', 'Genres']],
        on='MovieID',
        how='left'
    )
)

In [82]:
top_n_recommendations = (
    user_movie_scores_df
    .sort_values('Weighted Score', ascending=False)
    .head(10)
)
top_n_recommendations

,MovieID,Simple Score,Weighted Score,Title,Genres
894,2188,0.828052,4.225737,54 (1998),Drama
42,94,0.828052,4.225737,Beautiful Girls (1996),Drama
19,36,0.828052,4.225737,Dead Man Walking (1995),Drama
23,43,0.828052,4.225737,Restoration (1995),Drama
879,2147,0.828052,4.225737,"Clan of the Cave Bear, The (1986)",Drama
917,2269,0.828052,4.225737,Indecent Proposal (1993),Drama
851,2114,0.828052,4.225737,"Outsiders, The (1983)",Drama
856,2120,0.828052,4.225737,Needful Things (1993),Drama|Horror
864,2132,0.828052,4.225737,Who's Afraid of Virginia Woolf? (1966),Drama
933,2312,0.828052,4.225737,Children of a Lesser God (1986),Drama


In [89]:
def recommend_for_user(user_id, train_user_genre_profile_df, candidate_movies_df, train_df, top_n=10):
    user_preference = (
        train_user_genre_profile_df[
            train_user_genre_profile_df['UserID'] == user_id
        ]
        .sort_values(
            by='Preference Score Scaled',
            ascending=False
        )
    )
    user_train_rated_movies = train_df.loc[
        train_df['UserID'] == user_id,
        'MovieID'
    ].unique()
    user_candidates_df = candidate_movies_df[
        ~candidate_movies_df['MovieID'].isin(user_train_rated_movies)
    ].copy()
    user_movie_preference_df = (
        user_candidates_df
        .assign(Genres=user_candidates_df['Genres'].str.split('|'))
        .explode('Genres')
    )
    user_movie_preference_df = user_movie_preference_df.merge(
        user_preference[
            ['Genres', 'Preference Score', 'Preference Score Scaled']
        ],
        on='Genres',
        how='left'
    )
    simple_movie_score = (
        user_movie_preference_df
        .groupby('MovieID')['Preference Score Scaled']
        .mean()
    )
    user_movie_preference_df['Final Score'] = (
        2
        * user_movie_preference_df['Preference Score Scaled']
        * user_movie_preference_df['Weighted Rating Scaled']
        /
        (
            user_movie_preference_df['Preference Score Scaled']
            + user_movie_preference_df['Weighted Rating Scaled']
        )
    )
    movie_scores_df = (
        user_movie_preference_df
        .groupby('MovieID')
        .agg(
            Simple_Score=(
                'Preference Score Scaled',
                'mean'
            ),
            Final_Score=(
                'Final Score',
                'mean'
            )
        )
        .reset_index()
    )
    movie_scores_df = (
        movie_scores_df
        .merge(
            user_candidates_df[
                [
                    'MovieID',
                    'Title',
                    'Genres',
                    'Average_Rating',
                    'Rating_Count',
                    'Weighted_Rating'
                ]
            ],
            on='MovieID',
            how='left'
        )
    )
    top_n_recommendations = (
        movie_scores_df
        .sort_values('Final_Score', ascending=False)
        .head(top_n)
    )
    return top_n_recommendations

In [90]:
recommendations_user_1 = recommend_for_user(
    user_id=1,
    train_user_genre_profile_df=train_user_genre_profile_df,
    candidate_movies_df=candidate_movies_df,
    train_df=train_df,
    top_n=10
)
recommendations_user_1

,MovieID,Simple_Score,Final_Score,Title,Genres,Average_Rating,Rating_Count,Weighted_Rating
306,318,0.828052,0.905939,"Shawshank Redemption, The (1994)",Drama,4.565570,1975,4.523360
839,923,0.828052,0.871159,Citizen Kane (1941),Drama,4.401053,950,4.331848
1790,2019,0.778953,0.860913,Seven Samurai (The Magnificent Seven) (Shichin...,Action|Drama,4.582857,525,4.438874
782,858,0.748466,0.849493,"Godfather, The (1972)",Action|Crime|Drama,4.528770,2016,4.488987
868,953,0.828052,0.848296,It's a Wonderful Life (1946),Drama,4.312611,563,4.214940
1101,1225,0.828052,0.846766,Amadeus (1984),Drama,4.253165,1185,4.207350
1123,1250,0.787608,0.846088,"Bridge on the River Kwai, The (1957)",Drama|War,4.407843,765,4.322970
1080,1203,0.828052,0.845687,12 Angry Men (1957),Drama,4.310484,496,4.202016
2585,2858,0.783651,0.841078,American Beauty (1999),Comedy|Drama,4.328607,3174,4.308569
2075,2324,0.783651,0.836643,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama,4.344455,1019,4.284235


The new scoring strategy produces more differentiated and diverse recommendations because it incorporates both user preference and movie-level quality.

In [107]:
user_test_relevant = test_relevant_df[
    test_relevant_df['UserID'] == user_id
]

user_test_relevant

,UserID,MovieID,Rating,Timestamp,TimestampDate,YearMonth
33,1,588,4,978824268,2001-01-06 23:37:48,2001-01
40,1,1,5,978824268,2001-01-06 23:37:48,2001-01
4,1,2355,5,978824291,2001-01-06 23:38:11,2001-01
30,1,2294,4,978824291,2001-01-06 23:38:11,2001-01
35,1,783,4,978824291,2001-01-06 23:38:11,2001-01
32,1,1566,4,978824330,2001-01-06 23:38:50,2001-01
34,1,1907,4,978824330,2001-01-06 23:38:50,2001-01
25,1,48,5,978824351,2001-01-06 23:39:11,2001-01


In [108]:
relevant_movie_ids = set(
    user_test_relevant['MovieID']
)

In [109]:
recommended_movie_ids = set(
    recommendations_user_1['MovieID']
)

In [110]:
relevant_recommendations = (
    recommended_movie_ids
    & relevant_movie_ids
)

In [111]:
relevant_count = len(relevant_recommendations)

relevant_count

0

In [114]:
precision = (
    relevant_count
    / len(recommended_movie_ids)
)

recall = (
    relevant_count
    / len(relevant_movie_ids)
)
precision, recall

(0.0, 0.0)

In [115]:
if precision + recall == 0:
    f1 = 0
else:
    f1 = (
        2 * precision * recall
        / (precision + recall)
    )

In [117]:
print("Recommended:", recommended_movie_ids)
print("Relevant:", relevant_movie_ids)
print("Intersection:", relevant_recommendations)

Recommended: {1250, 2019, 1225, 2858, 1203, 2324, 953, 858, 923, 318}
Relevant: {1, 588, 783, 48, 2355, 1907, 2294, 1566}
Intersection: set()


In [118]:
print("Recommended count:", len(recommended_movie_ids))
print("Relevant count:", len(relevant_movie_ids))
print("Relevant recommended:", len(relevant_recommendations))

Recommended count: 10
Relevant count: 8
Relevant recommended: 0


In [119]:
print(f'Precision@10 for user-1: {precision}')
print(f'Recall@10 for user-1: {recall}')
print(f'F1@10 for user-1: {f1}')

Precision@10 for user-1: 0.0
Recall@10 for user-1: 0.0
F1@10 for user-1: 0


In [125]:
def evaluate_recommendations(user_id, train_user_genre_profile_df, candidate_movies_df, train_df, test_relevant_df, top_n=10):
    recommendations = recommend_for_user(
        user_id,
        train_user_genre_profile_df,
        candidate_movies_df,
        train_df,
        top_n=top_n
    )
    user_test_relevant = test_relevant_df[
        test_relevant_df['UserID'] == user_id
    ]
    relevant_movie_ids = set(
        user_test_relevant['MovieID']
    )
    recommended_movie_ids = set(
        recommendations['MovieID']
    )
    relevant_recommendations = (
        recommended_movie_ids
        & relevant_movie_ids
    )
    relevant_count = len(
        relevant_recommendations
    )
    # 5. Calculate Precision@K
    precision = (
        relevant_count
        / len(recommended_movie_ids)
        if len(recommended_movie_ids) > 0
        else 0
    )

    # 6. Calculate Recall@K
    recall = (
        relevant_count
        / len(relevant_movie_ids)
        if len(relevant_movie_ids) > 0
        else 0
    )

    # 7. Calculate F1@K
    if precision + recall == 0:
        f1 = 0
    else:
        f1 = (
            2 * precision * recall
            / (precision + recall)
        )

    return {
        'UserID': user_id,
        f'Precision@{top_n}': precision,
        f'Recall@{top_n}': recall,
        f'F1@{top_n}': f1
    }

In [126]:
user_1_evaluation = evaluate_recommendations(
    user_id=1,
    train_user_genre_profile_df=train_user_genre_profile_df,
    candidate_movies_df=candidate_movies_df,
    train_df=train_df,
    test_relevant_df=test_relevant_df,
    top_n=10
)

user_1_evaluation

{'UserID': 1, 'Precision@10': 0.0, 'Recall@10': 0.0, 'F1@10': 0}

It returned the same results as before and it looks generalized, but for checking the performance of function we make tests on another user too.

In [127]:
user_100_evaluation = evaluate_recommendations(
    user_id=100,
    train_user_genre_profile_df=train_user_genre_profile_df,
    candidate_movies_df=candidate_movies_df,
    train_df=train_df,
    test_relevant_df=test_relevant_df,
    top_n=10
)

user_100_evaluation

{'UserID': 100,
 'Precision@10': 0.3,
 'Recall@10': 0.5,
 'F1@10': 0.37499999999999994}

In [128]:
evaluation_results = []

for user_id in evaluation_users:
    result = evaluate_recommendations(
        user_id=user_id,
        train_user_genre_profile_df=train_user_genre_profile_df,
        candidate_movies_df=candidate_movies_df,
        train_df=train_df,
        test_relevant_df=test_relevant_df,
        top_n=10
    )
    
    evaluation_results.append(result)

evaluation_results_df = pd.DataFrame(evaluation_results)

evaluation_results_df.head()

,UserID,Precision@10,Recall@10,F1@10
0,1,0.0,0.0,0.0
1,2,0.0,0.0,0.0
2,3,0.0,0.0,0.0
3,4,0.0,0.0,0.0
4,5,0.0,0.0,0.0


In [131]:
evaluation_results_df.sample(10)

,UserID,Precision@10,Recall@10,F1@10
5381,5460,0.0,0.000000,0.000000
1613,1644,0.1,0.166667,0.125000
3439,3489,0.0,0.000000,0.000000
5303,5382,0.0,0.000000,0.000000
5623,5703,0.0,0.000000,0.000000
4097,4156,0.5,0.080645,0.138889
5008,5080,0.0,0.000000,0.000000
517,524,0.5,0.079365,0.136986
1697,1731,0.0,0.000000,0.000000
919,933,0.0,0.000000,0.000000


Recommendation quality varies substantially across users.

In [129]:
overall_metrics = (
    evaluation_results_df[
        ['Precision@10', 'Recall@10', 'F1@10']
    ]
    .mean()
)

overall_metrics

Precision@10    0.045064
Recall@10       0.028039
F1@10           0.028109
dtype: float64

In [133]:
evaluation_results_df[
    ['Precision@10', 'Recall@10', 'F1@10']
].describe()

,Precision@10,Recall@10,F1@10
count,5956.000000,5956.000000,5956.000000
mean,0.045064,0.028039,0.028109
std,0.095230,0.073229,0.058474
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,0.100000,0.017857,0.030769
max,0.900000,1.000000,0.461538


The recommendation system achieved low Top-10 Precision, Recall, and F1 scores under the current evaluation setup.

## Key Findings

- A time-based train/test split was used to simulate a realistic recommendation scenario, with each user's later ratings reserved for evaluation.
- User genre preferences were estimated from training data using a confidence-weighted preference score that combines the user's genre-specific mean with the overall training mean.
- Movie quality was estimated separately using rating averages and rating counts, with Bayesian-style shrinkage toward the global mean to reduce the effect of small sample sizes.
- Personalized movie scores combined user genre preferences with overall movie quality, allowing the system to balance personalization and general movie quality.
- The final recommendation pipeline generated Top-10 recommendations while excluding movies already rated by the user in the training data.
- Evaluation was performed using a relevance threshold of 4 and manually implemented Precision@10, Recall@10, and F1@10.
- The overall mean performance was low: Precision@10 = 0.045, Recall@10 = 0.028, and F1@10 = 0.028.
- The median Precision@10, Recall@10, and F1@10 were all 0, indicating that more than half of the evaluated users had no relevant movies retrieved in their Top-10 recommendations.
- Performance varied substantially across users. The maximum Precision@10 reached 0.90 and the maximum Recall@10 reached 1.00, showing that the system performed considerably better for some users than for others.
- The low overall scores should be interpreted in the context of sparse user-item interactions and the strict Top-10 future-rating matching setup rather than being treated as evidence that the recommendation approach is universally ineffective.


## Sprint Retrospective

### What Went Well

* The recommendation pipeline was developed incrementally, allowing each component to be tested before moving to the next stage.
* The separation between training data, candidate generation, recommendation, and evaluation helped prevent test-data leakage.
* Confidence-weighted genre preferences and movie-level quality scores provided a more robust basis for recommendations than simple averages alone.
* The evaluation process was implemented from scratch, which helped clarify how Precision, Recall, and F1 measure recommendation quality.

### What Was Challenging

* Designing a fair scoring method for movies with multiple genres required careful consideration of how genre preferences should be aggregated.
* Separating user preference from overall movie quality was important but introduced additional decisions about weighting and normalization.
* The sparse nature of user-item interactions made future-item prediction difficult and resulted in low overall Top-10 evaluation scores.
* Understanding the distinction between recommendations, predictions, and ground truth was an important part of building the evaluation pipeline correctly.

### What We Learned

* Recommendation systems require a balance between personalization and general item quality.
* Rating counts are important when interpreting averages because ratings based on very small samples are less reliable.
* Evaluation should be performed on unseen future interactions rather than on the same data used to construct user preferences.
* A single user's evaluation cannot represent the performance of the entire recommendation system.
* Precision and Recall capture different aspects of recommendation quality, so neither should be interpreted in isolation.

### What Could Be Improved

* The recommendation model could incorporate additional user-item signals beyond genre preferences and global movie quality.
* More advanced similarity or collaborative filtering approaches could help identify users and movies with similar rating patterns.
* Evaluation could be extended to multiple values of K rather than relying only on Top-10.
* Additional baselines would make it easier to determine whether the current approach provides meaningful improvement over simpler recommendation strategies.

### Next Steps

* Compare the current approach against simple popularity-based and rating-based baselines.
* Explore additional recommendation signals and user-item similarity.
* Evaluate the system across multiple values of K.
* Analyze which user groups benefit most from the current recommendation strategy.
